This analysis creates a presentation-ready information card for Valve using the Steam dataset.

The card reports:

- number of Valve games;
- average and median price;
- dominant genre;
- game with the highest positive-rating percentage.

The Publisher Profile is generated from the computed results so that its interpretation remains grounded in the dataset.

In [1]:
import pandas as pd


steam = pd.read_csv("data/steam/steam.csv")

required_columns = {
    "appid",
    "name",
    "publisher",
    "genres",
    "price",
    "positive_ratings",
    "negative_ratings",
}

missing = required_columns - set(steam.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )


steam["price"] = pd.to_numeric(
    steam["price"],
    errors="coerce",
)

steam["positive_ratings"] = pd.to_numeric(
    steam["positive_ratings"],
    errors="coerce",
)

steam["negative_ratings"] = pd.to_numeric(
    steam["negative_ratings"],
    errors="coerce",
)


publisher_labels = (
    steam["publisher"]
    .fillna("")
    .astype(str)
    .str.split(";")
)

valve_mask = publisher_labels.apply(
    lambda labels: any(
        label.strip().casefold() == "valve"
        for label in labels
    )
)

valve_games = steam.loc[valve_mask].copy()

if valve_games.empty:
    raise ValueError(
        "No Valve games were found in the dataset."
    )


game_count = valve_games["appid"].nunique()
mean_price = valve_games["price"].mean()
median_price = valve_games["price"].median()


genre_counts = (
    valve_games[
        ["appid", "genres"]
    ]
    .assign(
        genre=(
            valve_games["genres"]
            .fillna("")
            .str.split(";")
        )
    )
    .explode("genre")
)

genre_counts["genre"] = (
    genre_counts["genre"]
    .str.strip()
)

genre_counts = (
    genre_counts[
        genre_counts["genre"].ne("")
    ]
    .drop_duplicates(
        ["appid", "genre"]
    )
    .groupby("genre")["appid"]
    .nunique()
    .sort_values(ascending=False)
)

dominant_genre = genre_counts.index[0]
dominant_genre_count = int(
    genre_counts.iloc[0]
)


valve_games["total_ratings"] = (
    valve_games["positive_ratings"]
    + valve_games["negative_ratings"]
)

rated_games = (
    valve_games[
        valve_games["total_ratings"] > 0
    ]
    .copy()
)

rated_games["positive_percentage"] = (
    rated_games["positive_ratings"]
    / rated_games["total_ratings"]
    * 100
)

featured_game = (
    rated_games
    .sort_values(
        [
            "positive_percentage",
            "total_ratings",
        ],
        ascending=False,
    )
    .iloc[0]
)


print("Valve games:", game_count)
print("Average price:", f"${mean_price:.2f}")
print("Median price:", f"${median_price:.2f}")
print(
    "Dominant genre:",
    dominant_genre,
    f"({dominant_genre_count} games)",
)
print(
    "Highest positive-rating percentage:",
    featured_game["name"],
    f"({featured_game['positive_percentage']:.2f}%)",
)

Valve games: 30
Average price: $4.48
Median price: $3.99
Dominant genre: Action (26 games)
Highest positive-rating percentage: Portal 2 (98.65%)


In [2]:
from html import escape
from IPython.display import HTML, display


if mean_price > median_price:
    pricing_text = (
        f"The average price (${mean_price:.2f}) is above the "
        f"median (${median_price:.2f}), indicating that some "
        "higher-priced titles raise the overall average."
    )
elif mean_price < median_price:
    pricing_text = (
        f"The average price (${mean_price:.2f}) is below the "
        f"median (${median_price:.2f}), indicating that lower-priced "
        "titles reduce the overall average."
    )
else:
    pricing_text = (
        f"The average and median prices are both ${mean_price:.2f}, "
        "indicating a relatively centered price distribution."
    )


genre_share = (
    dominant_genre_count
    / game_count
)

if genre_share >= 0.50:
    portfolio_text = (
        f"The portfolio is concentrated in {dominant_genre}, "
        f"which accounts for {genre_share:.0%} of Valve games."
    )
else:
    portfolio_text = (
        f"{dominant_genre} is the most common genre, but it accounts "
        f"for only {genre_share:.0%} of Valve games, suggesting that "
        "the portfolio spans multiple genres."
    )


reception_text = (
    f"{featured_game['name']} has the highest positive-rating "
    f"percentage at {featured_game['positive_percentage']:.2f}% "
    f"across {int(featured_game['total_ratings']):,} ratings."
)


publisher_profile = (
    f"{pricing_text} "
    f"{portfolio_text} "
    f"{reception_text}"
)


card_html = f"""
<div style="
    max-width:760px;
    border:1px solid #d0d0d0;
    border-radius:16px;
    overflow:hidden;
    font-family:Arial,sans-serif;
">
    <div style="
        background:#20252b;
        color:white;
        padding:20px 24px;
    ">
        <h2 style="margin:0;">Valve Publisher Profile</h2>
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">
        <tr style="background:#f3f3f3;">
            <td style="padding:12px 20px;"><b>Number of games</b></td>
            <td style="padding:12px 20px;">{game_count}</td>
        </tr>
        <tr>
            <td style="padding:12px 20px;"><b>Average price</b></td>
            <td style="padding:12px 20px;">${mean_price:.2f}</td>
        </tr>
        <tr style="background:#f3f3f3;">
            <td style="padding:12px 20px;"><b>Median price</b></td>
            <td style="padding:12px 20px;">${median_price:.2f}</td>
        </tr>
        <tr>
            <td style="padding:12px 20px;"><b>Dominant genre</b></td>
            <td style="padding:12px 20px;">
                {escape(dominant_genre)}
                ({dominant_genre_count} games)
            </td>
        </tr>
    </table>

    <div style="
        margin:18px 24px;
        padding:16px;
        border-radius:10px;
        background:#e8f5e9;
        border-left:5px solid #2e7d32;
    ">
        <b>Featured Game</b><br>
        {escape(str(featured_game["name"]))}<br>
        {featured_game["positive_percentage"]:.2f}% positive ratings
    </div>

    <div style="padding:0 24px 20px;">
        <h3>Publisher Profile</h3>
        <p style="line-height:1.6;">
            {escape(publisher_profile)}
        </p>
    </div>

    <div style="
        background:#eeeeee;
        color:#777777;
        padding:10px 24px;
        font-size:12px;
        font-style:italic;
        text-align:right;
    ">
        Source: Steam Store Games dataset
    </div>
</div>
"""


display(
    HTML(card_html)
)

Number of games,30
Average price,$4.48
Median price,$3.99
Dominant genre,Action (26 games)


The card uses only computed dataset results.

Its pricing interpretation depends on the observed mean and median, its portfolio description depends on the dominant genre's share of Valve games, and its player-reception statement reports the featured game's calculated positive-rating percentage and rating count.